# Nicha's song-gpt

### reading and exploring the data

- download ur datasets
- open input.txt and read all text as a string
- check if the first 1000 characters are correct
- chars = sorted(list(set(text))) → create a sorted list of chars
- num(chars) → vocabulary n

In [64]:
import os

all_text = ""

lyrics = "./data/data/Albums"

for root, dir, files in os.walk(lyrics):
        for file in files:
            if file.endswith(".txt"):
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    all_text += f.read()
                    all_text += "\n\n"

# write lyrics into combined folder
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

# read the combined text file and print the first 1000 characters
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

125 ContributorsTranslationsEspañolPortuguêsItalianoDeutschFrançaisTürkçeСрпски中文MagyarPolskiУкраїнська​evermore Lyrics[Verse 1: Taylor Swift]
Gray November
I've been down since July
Motion capture
Put me in a bad light
I replay my footsteps on each stepping stone
Trying to find the one where I went wrong
Writing letters
Addressed to the fire

[Chorus: Taylor Swift]
And I was catching my breath
Staring out an open window
Catching my death
And I couldn't be sure
I had a feeling so peculiar
That this pain would be for
Evermore
[Verse 2: Taylor Swift]
Hey December
Guess I'm feeling unmoored
Can't remember
What I used to fight for
I rewind thе tape, but all it does is pause
On thе very moment all was lost
Sending signals
To be double-crossed

[Chorus: Taylor Swift & Justin Vernon]
And I was catching my breath
Barefoot in the wildest winter
Catching my death
And I couldn't be sure
I had a feeling so peculiar
That this pain would be for
Evermore
(Evermore)
[Bridge: Justin Vernon, Taylor Swif

In [65]:
chars = sorted(list(set(text))) # vocab list -> all the possible letters the model can predict
vocab_size = len(chars)

print("vocab size:", vocab_size)

vocab size: 202


### tokenisation, train/val split

- tokenise = convert raw text as a string to some sequence of integers
- create a mapping from characters to integers → for encoder decoder
- encoder: take a string, output a list of integers
- decoder: take a list of integers, output a string
- gpt uses tiktoken
- encode entire text dataset and store it into a torch.Tensor
- separate into train/val split

In [66]:
import torch
import numpy

# char <-> int mapping 
char_to_int = {char:int for int, char in enumerate(chars)}
int_to_char = {int:char for int, char in enumerate(chars)}

# encode the text into integers
def encode(text):
    return [char_to_int[char] for char in text]

# decode the integers back into text
def decode(text):
    return "".join([int_to_char[int] for int in text])

encoded_text = encode(text)
data = torch.tensor(encoded_text, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([2114880]) torch.int64
tensor([ 17,  18,  21,   2,  31,  72,  71,  77,  75,  66,  59,  78,  77,  72,
         75,  76,  48,  75,  58,  71,  76,  69,  58,  77,  66,  72,  71,  76,
         33,  76,  73,  58, 100,  72,  69,  44,  72,  75,  77,  78,  64,  78,
         97,  76,  37,  77,  58,  69,  66,  58,  71,  72,  32,  62,  78,  77,
         76,  60,  65,  34,  75,  58,  71,  95,  58,  66,  76,  48, 103,  75,
         68,  95,  62, 125, 138, 137, 139, 133, 131, 192, 193,  41,  58,  64,
         82,  58,  75,  44,  72,  69,  76,  68,  66, 126, 133, 138, 127, 142,
        135, 139, 141, 133, 127, 180,  62,  79,  62,  75,  70,  72,  75,  62,
          2,  40,  82,  75,  66,  60,  76,  55,  50,  62,  75,  76,  62,   2,
         17,  26,   2,  48,  58,  82,  69,  72,  75,   2,  47,  80,  66,  63,
         77,  56,   1,  35,  75,  58,  82,   2,  42,  72,  79,  62,  70,  59,
         62,  75,   1,  37,   8,  79,  62,   2,  59,  62,  62,  71,   2,  61,
         72,  80,  71,   2,  7

In [67]:
# train/val split
n = int(0.9*len(data))
train_data = data[:n] # first 90% of data for training
val_data = data[n:] # last 10% of data for validation

### data loader: batches of chunks of data

- we do not feed the entire text into the transformer all at once → computationally very expensive and prohibitive
    - we work with chunks of data at a time = block size (max length)
    - eg: tensor([18, 47, 56, 57, 58, 1, 15, 47, 58])
    - when input is tensor([18]) the target: 47
    - when input is tensor([18, 47]) the target: 56 etc
    - you want the transformer to see everything

In [68]:
torch.manual_seed(1337)
block_size = 8 # size of chunk model looks at
batch_size = 4 # num of chunks to process in parallel
n_embd = 32 # dimensionality of character embedding


# (4,) = vector with 4 elements 
def get_batch(split):
    data = train_data if split == "train" else val_data
    starting_pos = torch.randint(len(data) - block_size, (batch_size,)) 
    x = torch.stack([data[pos:pos+block_size] for pos in starting_pos])
    y = torch.stack([data[pos+1:pos+block_size+1] for pos in starting_pos])
    return x, y

xb, yb = get_batch("train")

### Self Attention 
- bigram: each token only looks at itself to predict the next token
- self-attention: lets every token look at all previous tokens and decide which ones are most relevant 
#### intuition
- each token asks: which past tokens should i pay attention to?
- every token produces 3 vectors:
    - query(Q): what am i looking for?
    - key(K): what do i contain?
    - value(V): what do i communicate if attended to?
- attention score between 2 tokens = how much their Q and K agree (dot pdt), high score = pay more attention to that token's V 
- one set of Q, K, V = 1 head
#### math
Attention(Q, K, V) = softmax(QKᵀ / √d_k) * V
- QKᵀ — score every token against every other token
- / √d_k — scale down to prevent softmax from becoming too sharp
- softmax — convert scores to probabilities (weights)
- * V — weighted sum of values = the output
#### masking (causal attention)
- during training you can't let token 3 look at token 5 because in generation token 5 doesn't exist yet
- set future scores to -inf before softmax -> make them 0 after softmax

In [69]:
import torch.nn as nn
from torch.nn import functional as F

n_heads = 4
head_size = n_embd // n_heads  # 8

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        scores = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        scores = F.softmax(scores, dim=-1)
        v = self.value(x)
        return scores @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)


### feed-forward neural network
- attention lets tokens communicate with each other — token 3 can look at tokens 1 and 2
- but after that, each token only has a weighted sum of other tokens' values
- it hasn't had a chance to individually think about what it received.
- the feed-forward layer is where each token processes its own information independently

#### structure
x → Linear(n_embd, 4*n_embd) → ReLU → Linear(4*n_embd, n_embd)
- expand to 4*n_embd — gives the network more room to compute
- ReLU — non-linearity so it can learn complex patterns, not just linear transforms
- project back to n_embd — back to the same size so it plugs back in

In [70]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential( # chains the layers 
            nn.Linear(n_embd, 4*n_embd), # expand
            nn.ReLU(), # activation fxn
            nn.Linear(4*n_embd, n_embd), # project back to og embedding size
        )
    
    def forward(self, x):
        return self.net(x)

### simplest baseline: bigram language model, loss, generation

- bigram model is the simplest possible language model — predicts the next character based on only the current character, no memory, no context just: given i see e, what usually comes next?
- a token is a character that is mapped to a number then after that u give each number n_embd # of numbers -> embedding layer
- forward() means: how data flows through the neural network, defines input → computation → output
- generate() means: how you use the model to produce text after training, give it a starting    sequence of tokens, and it keeps predicting and appending one new token at a time until it hits max_new_tokens   

In [71]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.multi_head_attention = MultiHeadAttention(n_heads, head_size)
        self.feed_forward = FeedForward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, input, targets=None):
        x = self.token_embedding_table(input) # (B, T, n_embd)
        x = x + self.multi_head_attention(self.ln1(x)) # (B, T, n_embd)
        x = x + self.feed_forward(self.ln2(x)) # (B, T, n_embd)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))

        return logits, loss

    def generate(self, input, max_new_tokens):
        for _ in range(max_new_tokens):
            input_cropped = input[:, -block_size:] # crop to block_size
            logits, _ = self(input_cropped)
            last_logits = logits[:, -1, :]  # (B, vocab_size)
            probabilities = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            input = torch.cat((input, next_token), dim=1)
        return input

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(loss)

tensor(5.4821, grad_fn=<NllLossBackward0>)


### training loop
- optimizer: algorithm that adjusts the model's weights after each step
- AdamW specifically is an improved version of the classic gradient descent -> instead of updating all weights by the same step size, it adapts the step size per parameter based on how much that parameter has been changing
- order: forward -> zero grad -> backward -> step

In [72]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for steps in range(10000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True) # clear gradients from previous step
    loss.backward() # compute gradients via backpropagation
    optimizer.step() # update weights that have been computed in backpropagation -> actual learning happens

print(loss.item())

2.530831813812256


what the model has learnt so far:

In [73]:
context = torch.zeros((1, 1), dtype=torch.long) # start with a single token -> starting seed
generated = model.generate(context, max_new_tokens=1000)[0].tolist()
print(decode(generated))

	“Porug
Gard cucand
Goug weyouright's d: Lonin' ar hinole tof Mandealtitie war, ayou yoou'm wnnowith l mee fe dgifre bustrup "PoBr my courigow yod I toun o t, tooyhe ldove bartongrttin'thi. (Rewrtwhis ank onow, drincevesy amind hire dat yedou doun't y ireafof but he go thand lerewhas boroknd d anloug
(Porp
Hands we (Gtoon-ur es
Oosby aye, whe wasedr NFrobuêmiak dind dodome be cive won ghoold and cld
'dind ce che lind
He'tre che panon't, I'm s, m a Souy thit do anig s tathere ink
An)E5Dorexp” watonden s, I tut Laingoong me中Bree7foug'ld linghes sciУкyong | Buthtomers bee nt otand gounot juld adugr was, atУкly cahabould
Ar be winesiffr
As, cSotr
1.
Bhave Rowllik re wishe were tofem (Re we
Gunow? RU1]
A
Itck bulunever in k f Ifr mo dorknand llifous anstiongs y, at I f he cushe min llad” WhiEmbrelat-'stren If kanthave coouncoy sese tet, ndo thacofre gh whe
(EmilFung bay, knou amitlo nght youcr houng coost be he por cuthe he muind th-d watrthen'lit d ing inonertoknkçe’ icimyl I's Bandy a gho

### residual connections
- as you stack more layers, gradients can vanish or explode during backprop — the signal gets too weak to update early layers
- residual connections fix this by adding the input directly to the output of each sublayer:

x = x + self.multi_head_attention(x)  # instead of x = self.multi_head_attention(x)

x = x + self.feed_forward(x)

- gives gradients a "highway" to flow back through — they can skip layers entirely via the + 
- deep networks become much easier to train

### layer norm
- normalizes each token's values to have mean 0 and std 1 before passing through attention/ffwd 
- keeps activations in a stable range so training doesn't blow up

self.ln1 = nn.LayerNorm(n_embd)

self.ln2 = nn.LayerNorm(n_embd)
